In [1]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data Historis
df_train = pd.read_csv('../data/processed/train.csv')

# 2. Load User Profile (Demografi dari MovieLens)
user_cols = ['user_id', 'age', 'gender', 'occupation', 'zip_code']
df_users = pd.read_csv('../data/raw/ml-100k/u.user', sep='|', names=user_cols)

# 3. Feature Engineering: Membuat statistik sederhana (Item Popularity & User Activity)
item_stats = df_train.groupby('item_id').agg(
    item_pop=('rating', 'count'),
    item_avg_rating=('rating', 'mean')
).reset_index()

user_stats = df_train.groupby('user_id').agg(
    user_activity=('rating', 'count'),
    user_avg_rating=('rating', 'mean')
).reset_index()

# 4. Menggabungkan semua fitur menjadi satu tabel (Master Table)
df_features = df_train.merge(df_users[['user_id', 'age', 'gender', 'occupation']], on='user_id')
df_features = df_features.merge(item_stats, on='item_id')
df_features = df_features.merge(user_stats, on='user_id')

# 5. Mendefinisikan Target Prediksi (Simulasi Implicit Feedback: Rating >= 4 dianggap Klik/Beli = 1, sisanya = 0)
df_features['target'] = (df_features['rating'] >= 4).astype(int)

# Konversi tipe data kategori untuk LightGBM
for col in ['gender', 'occupation']:
    df_features[col] = df_features[col].astype('category')

print(f"Data siap! Total baris pelatihan: {len(df_features)}")
display(df_features[['user_id', 'item_id', 'age', 'gender', 'item_pop', 'target']].head())

Data siap! Total baris pelatihan: 80000


,user_id,item_id,age,gender,item_pop,target
0,259,255,21,M,139,1
1,259,286,21,M,370,1
2,259,298,21,M,162,1
3,259,185,21,M,192,1
4,259,173,21,M,266,1


In [2]:
# Tentukan fitur yang akan dipelajari AI (tanpa ID)
features = ['age', 'gender', 'occupation', 'item_pop', 'item_avg_rating', 'user_activity', 'user_avg_rating']
X = df_features[features]
y = df_features['target']

# Pisahkan sedikit data untuk validasi internal model
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Buat format dataset LightGBM
lgb_train = lgb.Dataset(X_train, y_train)
lgb_eval = lgb.Dataset(X_val, y_val, reference=lgb_train)

# Konfigurasi Parameter (Objective 'binary' karena kita memprediksi 0 atau 1)
params = {
    'objective': 'binary',
    'metric': 'auc', # Area Under Curve (mengukur kemampuan model membedakan kelas)
    'learning_rate': 0.1,
    'verbose': -1
}

print("Memulai pelatihan model LightGBM...")
# Latih model dengan Early Stopping agar berhenti saat performa tidak lagi meningkat
model_rank = lgb.train(
    params, 
    lgb_train, 
    valid_sets=[lgb_train, lgb_eval], 
    num_boost_round=150, 
    callbacks=[lgb.early_stopping(stopping_rounds=15), lgb.log_evaluation(50)]
)
print("Pelatihan selesai!")

Memulai pelatihan model LightGBM...
Training until validation scores don't improve for 15 rounds
[50]	training's auc: 0.802693	valid_1's auc: 0.791256
[100]	training's auc: 0.813039	valid_1's auc: 0.79221
Early stopping, best iteration is:
[116]	training's auc: 0.816288	valid_1's auc: 0.792596
Pelatihan selesai!


In [3]:
def rank_candidates(user_id, candidate_list):
    # 1. Tarik profil user
    user_info = df_users[df_users['user_id'] == user_id].iloc[0]
    u_stat = user_stats[user_stats['user_id'] == user_id]
    
    # Ambil statistik, atau berikan nilai default jika user baru (Cold-Start)
    u_activity = u_stat['user_activity'].values[0] if not u_stat.empty else 0
    u_avg = u_stat['user_avg_rating'].values[0] if not u_stat.empty else 3.0
    
    # 2. Susun dataframe sementara (bayangkan ini terjadi di memori saat aplikasi berjalan)
    rank_df = pd.DataFrame({'item_id': candidate_list})
    rank_df['age'] = user_info['age']
    rank_df['gender'] = user_info['gender']
    rank_df['occupation'] = user_info['occupation']
    rank_df['user_activity'] = u_activity
    rank_df['user_avg_rating'] = u_avg
    
    # Gabungkan dengan statistik item
    rank_df = rank_df.merge(item_stats, on='item_id', how='left').fillna(0)
    
    # 3. Prediksi probabilitas interaksi positif menggunakan LightGBM
    rank_df['gender'] = rank_df['gender'].astype('category')
    rank_df['occupation'] = rank_df['occupation'].astype('category')
    
    X_pred = rank_df[features]
    rank_df['ranking_score'] = model_rank.predict(X_pred)
    
    # 4. Urutkan dan ambil Top 10 Final
    final_top_10 = rank_df.sort_values('ranking_score', ascending=False).head(10)
    
    return final_top_10[['item_id', 'ranking_score']]

# Uji coba dengan 15 ID Kandidat Acak (Simulasi output dari Phase 8)
sample_candidates = [50, 258, 100, 318, 127, 45, 99, 12, 1, 98, 300, 150, 500, 420, 333]
test_user = 12

print(f"Mengurutkan ulang kandidat untuk User {test_user}...\n")
hasil_ranking = rank_candidates(test_user, sample_candidates)
display(hasil_ranking)

Mengurutkan ulang kandidat untuk User 12...



,item_id,ranking_score
3,318,0.959928
12,500,0.952195
4,127,0.947646
0,50,0.947576
7,12,0.946668
9,98,0.939909
2,100,0.930426
5,45,0.925590
13,420,0.909236
14,333,0.897063
